# landing → bronze: 秋田県オープンデータの CSV 全部

SageMaker Unified Studio の JupyterLab(Python 3 カーネル)で上から順に実行する。
合計 26MB 程度(地震被害想定の 250m メッシュ 27 本を含めると約 3.4GB)なので Spark は使わず boto3 だけで処理する。
50MB を超えるファイルは変換した中身をメモリに残さず、書くときに landing から読み直して変換する(並列も2本まで)。

landing には `scripts/fetch_akita_opendata.py` で CKAN から落とした CSV が、
**加工せず** データセットごとに置いてある(`scripts/upload_opendata_landing.sh` で上げる)。
ここでは中身の形(見出しの位置・列数など)には手を付けず、**文字コードと改行だけ** をそろえる。
形の良し悪しは bronze を読む `opendata_bronze_report.ipynb` が評価する。

1. 文字コード: BOM 付き UTF-8 / UTF-8 / CP932(Shift_JIS) を判定して UTF-8(BOMなし)にする
2. 改行コード: CRLF と CR を LF に置き換える
3. CSV として読めるかを確かめる(読めないファイルは bronze に入れず report に残す)
4. 一時置き場 → 検証 → bronze を truncate してコピー、の3段階で書く

| 層 | 場所 |
| --- | --- |
| landing | `s3://<bucket>/akita/landing/opendata/<データセット>/<ファイル>.csv` |
| bronze | `s3://<bucket>/akita/bronze/opendata/<データセット>/<ファイル>.csv` |
| bronze の台帳 | `s3://<bucket>/akita/bronze/opendata/_manifest.json` |
| 確認結果 | `s3://<bucket>/akita/_reports/opendata/<実行時刻>.json` |
| 一時置き場 | `s3://<bucket>/akita/_staging/opendata/<実行時刻>/` |

年齢各歳(`population_by_age/`)と違い、1本でも失敗したら全部止めることはしない。
114 データセットは作り手も形もばらばらで、1本の不備で全体を止めると何も評価できないため。
失敗したファイルは bronze に入れず、理由を台帳と report に残す。

手元で試すとき(AWS に触らない): 環境変数 `OPENDATA_LOCAL_ROOT` にディレクトリを指定すると、
そこをバケットの代わりに読み書きする(`make opendata-local` が使う)。

出典: 秋田県オープンデータカタログサイト(https://ckan.pref.akita.lg.jp/), CC BY 4.0

In [ ]:
# ── 設定 ──────────────────────────────────────────────────────────────────
import os

# True のあいだは確認だけして書き込まない。結果を見てから False にして再実行する。
DRY_RUN = os.environ.get("DRY_RUN", "1") != "0"

REGION = "ap-northeast-1"
BUCKET_PREFIX = "jaws-festa-2026"
LANDING_PREFIX = "akita/landing/opendata/"
BRONZE_PREFIX = "akita/bronze/opendata/"
REPORT_PREFIX = "akita/_reports/opendata/"
STAGING_PREFIX = "akita/_staging/opendata/"
MANIFEST_NAME = "_manifest.json"

# 設定するとバケットの代わりにこのディレクトリを使う(手元での確認用)
LOCAL_ROOT = os.environ.get("OPENDATA_LOCAL_ROOT")

In [ ]:
# ── 読み書き先(S3 か、手元のディレクトリ) ──────────────────────────────────
from pathlib import Path


class S3Store:
    def __init__(self, bucket: str, client):
        self.bucket, self.s3 = bucket, client
        self.label = f"s3://{bucket}/"

    def keys(self, prefix: str) -> list[str]:
        return sorted(
            o["Key"]
            for page in self.s3.get_paginator("list_objects_v2").paginate(Bucket=self.bucket, Prefix=prefix)
            for o in page.get("Contents", [])
        )

    def get(self, key: str) -> bytes:
        return self.s3.get_object(Bucket=self.bucket, Key=key)["Body"].read()

    def put(self, key: str, body: bytes, content_type: str) -> None:
        self.s3.put_object(Bucket=self.bucket, Key=key, Body=body, ContentType=content_type)

    def copy(self, src: str, dst: str) -> None:
        self.s3.copy_object(Bucket=self.bucket, Key=dst, CopySource={"Bucket": self.bucket, "Key": src},
                            MetadataDirective="COPY")

    def delete(self, keys: list[str]) -> None:
        for i in range(0, len(keys), 1000):
            res = self.s3.delete_objects(Bucket=self.bucket,
                                         Delete={"Objects": [{"Key": k} for k in keys[i:i + 1000]], "Quiet": True})
            if res.get("Errors"):
                raise RuntimeError(f"削除に失敗: {res['Errors'][:3]}")


class LocalStore:
    def __init__(self, root: str):
        self.root = Path(root).resolve()
        self.label = f"{self.root}/"

    def keys(self, prefix: str) -> list[str]:
        out = []
        for d, _, files in os.walk(self.root, followlinks=True):
            for f in files:
                k = str(Path(d, f).relative_to(self.root))
                if k.startswith(prefix):
                    out.append(k)
        return sorted(out)

    def get(self, key: str) -> bytes:
        return (self.root / key).read_bytes()

    def put(self, key: str, body: bytes, content_type: str) -> None:
        p = self.root / key
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_bytes(body)

    def copy(self, src: str, dst: str) -> None:
        self.put(dst, self.get(src), "")

    def delete(self, keys: list[str]) -> None:
        for k in keys:
            (self.root / k).unlink()


if LOCAL_ROOT:
    store = LocalStore(LOCAL_ROOT)
else:
    import boto3

    session = boto3.Session(profile_name=os.environ.get("AWS_PROFILE"), region_name=REGION)
    account_id = session.client("sts").get_caller_identity()["Account"]
    # アカウントリージョン名前空間のバケット名(末尾 -an は必須サフィックス)
    store = S3Store(f"{BUCKET_PREFIX}-{account_id}-{REGION}-an", session.client("s3"))
print("読み書き先:", store.label, "/ DRY_RUN =", DRY_RUN)

In [ ]:
import json

landing_manifest = json.loads(store.get(LANDING_PREFIX + "manifest.json"))
keys = [k for k in store.keys(LANDING_PREFIX) if k.endswith(".csv")]
print(f"landing の CSV: {len(keys)} 本 / 取得時の内訳: {landing_manifest['summary']}"
      f"(取得日時 {landing_manifest['fetched_at']})")

# 台帳と実ファイルの突き合わせ。台帳に ok で載っているのに無い / 台帳に無いのにある、を出す
by_file = {r["file"]: r for r in landing_manifest["resources"]}
listed = {LANDING_PREFIX + r["file"] for r in landing_manifest["resources"] if r["status"] == "ok"}
missing, unlisted = sorted(listed - set(keys)), sorted(set(keys) - listed)
print(f"台帳にあって landing に無い: {len(missing)} 本 / landing にあって台帳に無い: {len(unlisted)} 本")
for k in (missing + unlisted)[:10]:
    print("  ", k)
assert keys, "landing に CSV が無い"

## 1〜3. 文字コード・改行コードをそろえて CSV として読めるか確かめる

- 文字コード: 先頭が BOM なら BOM 付き UTF-8。次に UTF-8 として読み、だめなら CP932 として読む。
  どれでも読めなければ失敗(変換しない)
- 改行コード: CRLF と CR を LF に置き換える。囲み `"` の中の改行も LF になる
- 中身は書き換えない。`csv.writer` で書き直すと囲み文字の付け方が変わり、landing との対応が
  取りにくくなるため。見出しの位置・注釈行などの評価はレポート側で行う
- 失敗とみなすもの: 文字コードが判定できない / 中身が HTML / NUL 文字を含む / 空 / `csv` で読めない

囲み文字 `"` は記録だけする。このレイヤーの bronze は Athena(LazySimpleSerDe)ではなく
ノートブックが直接読むので止める理由がない。Athena で読むなら OpenCSVSerDe が要る、という目印になる。

In [ ]:
import csv
import hashlib
import io
from concurrent.futures import ThreadPoolExecutor

BOM = b"\xef\xbb\xbf"
# これより大きいファイル(地震被害想定の 250m メッシュ、1本 120MB 前後)は、変換した中身をメモリに残さず、
# 書くときにもう一度 landing から読んで変換する。並列も少なくする(1本の変換に 1GB 近く使うため)
LARGE_BYTES = 50_000_000


def decode(raw: bytes) -> tuple[str, str]:
    if raw.startswith(BOM):
        return raw[3:].decode("utf-8"), "UTF-8 (BOMあり)"
    try:
        return raw.decode("utf-8"), "UTF-8"
    except UnicodeDecodeError:
        pass
    # Shift_JIS ではなく CP932 で読む(①や髙など、Windows の拡張文字が入っていることが多いため)
    return raw.decode("cp932"), "CP932"


def newline_kind(raw: bytes) -> str:
    crlf = raw.count(b"\r\n")
    lf = raw.count(b"\n") - crlf
    cr = raw.count(b"\r") - crlf
    kinds = [n for n, c in (("CRLF", crlf), ("LF", lf), ("CR", cr)) if c]
    return "+".join(kinds) or "なし"


def to_lf(text: str) -> str:
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    return text if text.endswith("\n") else text + "\n"


def pmap(fn, items: list[dict], size=lambda r: r["bytes_in"]) -> list:
    """順番を保ったまま並列に処理する。大きいファイルは2並列、それ以外は16並列。"""
    big = [i for i, r in enumerate(items) if size(r) > LARGE_BYTES]
    small = [i for i, r in enumerate(items) if size(r) <= LARGE_BYTES]
    out = [None] * len(items)
    for idx, workers in ((small, 16), (big, 2)):
        with ThreadPoolExecutor(max_workers=workers) as ex:
            for i, v in zip(idx, ex.map(lambda i: fn(items[i]), idx)):
                out[i] = v
    return out


def normalize(key: str) -> dict:
    raw = store.get(key)
    rel = key[len(LANDING_PREFIX):]
    meta = by_file.get(rel, {})
    r = {
        "file": rel,
        "dataset": rel.split("/", 1)[0],
        "resource_id": meta.get("resource_id"),
        "resource_name": meta.get("resource_name"),
        "last_modified": meta.get("last_modified"),
        "bytes_in": len(raw),
        "sha256_in": hashlib.sha256(raw).hexdigest(),
        "newline_in": newline_kind(raw),
        "problems": [],
    }
    if meta.get("sha256") and meta["sha256"] != r["sha256_in"]:
        r["problems"].append("landing の中身が取得時の台帳(sha256)と一致しない")
    try:
        text, r["encoding_in"] = decode(raw)
    except UnicodeDecodeError as e:
        r["encoding_in"] = "不明"
        r["problems"].append(f"UTF-8 でも CP932 でも読めない(変換しない): {e}")
        return r
    del raw
    head = text.lstrip()[:200].lower()
    if head.startswith(("<!doctype", "<html", "<?xml")):
        r["problems"].append("中身が HTML/XML(CSV ではない)")
    if "\x00" in text:
        r["problems"].append("NUL 文字を含む")
    if not text.strip():
        r["problems"].append("空のファイル")
    text = to_lf(text)
    n_rows = max_cols = 0
    try:
        for row in csv.reader(io.StringIO(text)):  # 行を溜めずに数える
            n_rows += 1
            max_cols = max(max_cols, len(row))
    except csv.Error as e:
        r["problems"].append(f"CSV として読めない: {e}")
    out = text.encode("utf-8")
    r.update(
        rows=n_rows,
        max_columns=max_cols,
        has_quote='"' in text,
        bytes_out=len(out),
        sha256_out=hashlib.sha256(out).hexdigest(),
        normalized=out if len(out) <= LARGE_BYTES else None,
    )
    return r


def body_of(r: dict) -> bytes:
    """bronze に書く中身。大きいファイルは landing から読み直して同じ変換をし、sha256 で同じことを確かめる。"""
    if r["normalized"] is not None:
        return r["normalized"]
    out = to_lf(decode(store.get(LANDING_PREFIX + r["file"]))[0]).encode("utf-8")
    assert hashlib.sha256(out).hexdigest() == r["sha256_out"], f"変換をやり直したら中身が変わった: {r['file']}"
    return out


sizes = {k: by_file.get(k[len(LANDING_PREFIX):], {}).get("bytes") or 0 for k in keys}
results = pmap(normalize, keys, size=lambda k: sizes[k])

ok = [r for r in results if not r["problems"]]
failed = [r for r in results if r["problems"]]


def tally(field: str, rows: list[dict]) -> dict:
    out: dict = {}
    for x in rows:
        out[x[field]] = out.get(x[field], 0) + 1
    return dict(sorted(out.items(), key=lambda kv: -kv[1]))


print("文字コード:", tally("encoding_in", results))
print("改行コード:", tally("newline_in", results))
print(f"囲み文字 \" を含む: {sum(1 for r in ok if r['has_quote'])} 本")
print(f"確認結果: OK {len(ok)} 本 / NG {len(failed)} 本"
      f"(うち {LARGE_BYTES / 1e6:.0f}MB 超 {sum(1 for r in ok if r['bytes_in'] > LARGE_BYTES)} 本)")
for r in failed:
    print("  NG", r["file"], r["problems"])

## 4. bronze に書き出す

年齢各歳の `landing_to_bronze.ipynb` と同じ3段階で書く。

1. **一時置き場に全部書く** — `_staging/opendata/<実行時刻>/`。bronze は触らない
2. **一時置き場を読み直して検証する** — そろっているか、中身が書いたものと一致するか
3. **bronze を truncate してからコピーする** — 最後に台帳 `_manifest.json` を書き、一時置き場を消す

truncate で、landing から消えたファイルの残りも bronze から消える。
1〜2 で失敗した場合、bronze は前回のまま。3 の途中で失敗した場合は、もう一度実行すれば揃う。
台帳を最後に書くので、台帳が新しければ bronze の中身もそろっている。

In [ ]:
from datetime import datetime, timezone

if not ok:
    raise SystemExit("bronze に書けるファイルが1本も無い")

ts = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
staging = f"{STAGING_PREFIX}{ts}/"
summary = [{k: v for k, v in r.items() if k != "normalized"} for r in results]

manifest = {
    "built_at": ts,
    "source": LANDING_PREFIX,
    "landing_fetched_at": landing_manifest["fetched_at"],
    "license": landing_manifest["license"],
    # データセットの名前・担当部署・CKAN の URL はレポートが見出しと出典に使う
    "datasets": {
        r["dataset"]: {k: r[k] for k in ("dataset_title", "dataset_url", "organization", "license")}
        for r in landing_manifest["resources"]
    },
    # 取得の段階で落としたもの(XLSX だった / 大きすぎて飛ばした)もレポートに出す
    "not_fetched": [
        {k: r.get(k) for k in ("dataset", "resource_name", "status", "detected", "size_declared")}
        for r in landing_manifest["resources"] if r["status"] != "ok"
    ],
    "files": [s for s in summary if not s["problems"]],
    "failed": [s for s in summary if s["problems"]],
}
report = {"run_at": ts, "store": store.label, "dry_run": DRY_RUN, "staging": staging, "files": summary}


def put_report(status: str, **extra) -> None:
    if DRY_RUN:
        return
    key = f"{REPORT_PREFIX}{ts}.json"
    store.put(key, json.dumps(report | {"status": status, **extra}, ensure_ascii=False, indent=1).encode(),
              "application/json")
    print(f"report: {store.label}{key}  status={status}")


# ── (1) 一時置き場に全部書く ────────────────────────────────────────────────
if DRY_RUN:
    print(f"[dry-run] {staging} に {len(ok)} 本({sum(r['bytes_out'] for r in ok):,} bytes)を書く")
else:
    pmap(lambda r: store.put(staging + r["file"], body_of(r), "text/csv; charset=utf-8"), ok)
    print(f"staged {len(ok)} 本 -> {store.label}{staging}")

In [ ]:
# ── (2) 一時置き場を読み直して検証する ──────────────────────────────────────
if not DRY_RUN:
    try:
        staged = [k[len(staging):] for k in store.keys(staging)]
        assert staged == sorted(r["file"] for r in ok), "一時置き場のファイルがそろっていない"
        # 読み直した中身を sha256 で突き合わせる(大きいファイルを2本ずつメモリに持たないため)
        back = pmap(lambda r: hashlib.sha256(store.get(staging + r["file"])).hexdigest(), ok)
        bad = [r["file"] for r, b in zip(ok, back) if b != r["sha256_out"]]
        assert not bad, f"中身が一致しない: {bad[:5]}"
    except Exception as e:
        put_report("failed_staging", error=str(e))
        raise SystemExit(f"一時置き場の検証に失敗。bronze は変更していない: {e}")
    print("一時置き場の検証 OK")

In [ ]:
# ── (3) bronze を truncate してからコピーし、台帳を書いて一時置き場を消す ──────
if DRY_RUN:
    print(f"[dry-run] truncate {store.label}{BRONZE_PREFIX}  {len(store.keys(BRONZE_PREFIX))} 本を消す")
    print(f"[dry-run] {len(ok)} 本をコピーして {BRONZE_PREFIX}{MANIFEST_NAME} を書く")
else:
    removed = store.keys(BRONZE_PREFIX)
    store.delete(removed)
    assert not store.keys(BRONZE_PREFIX), "truncate 後に bronze にファイルが残っている"
    report["truncated"] = len(removed)
    print(f"truncated {store.label}{BRONZE_PREFIX}  {len(removed)} 本")

    try:
        pmap(lambda r: store.copy(staging + r["file"], BRONZE_PREFIX + r["file"]), ok)
    except Exception as e:
        put_report("failed_copy", error=str(e))
        raise SystemExit(f"bronze へのコピーが途中で失敗。もう一度実行すれば揃う: {e}")

    back = pmap(lambda r: hashlib.sha256(store.get(BRONZE_PREFIX + r["file"])).hexdigest(), ok)
    bad = [r["file"] for r, b in zip(ok, back) if b != r["sha256_out"]]
    now = [k[len(BRONZE_PREFIX):] for k in store.keys(BRONZE_PREFIX)]
    if bad or now != sorted(r["file"] for r in ok):
        put_report("failed_verify", mismatched=bad)
        raise SystemExit(f"bronze の読み直しが一致しない: 中身 {bad[:5]} / 本数 {len(now)} (期待 {len(ok)})")
    print(f"bronze の読み直し OK({len(now)} 本)")

    store.put(BRONZE_PREFIX + MANIFEST_NAME, json.dumps(manifest, ensure_ascii=False, indent=1).encode(),
              "application/json")
    store.delete(store.keys(staging))
    put_report("succeeded")
    print(f"台帳: {store.label}{BRONZE_PREFIX}{MANIFEST_NAME}")